In [1]:
import os
import sys
sys.path.append('../..')
import utils

import panel as pn
import ollama

pn.extension()

In [2]:
def get_completion_from_messages(messages, model="llama3.2", temperature=0, max_tokens=500):
    """Generate response using Ollama's LLaMA 3.2 model."""
    formatted_messages = [{"role": msg["role"], "content": msg["content"]} for msg in messages]
    
    try:
        response = ollama.chat(model=model, messages=formatted_messages)
        return response['message']['content'].strip()  # Ensure clean output
    except Exception as e:
        print("ERROR: Ollama API call failed:", str(e))
        return "[]"  # Return empty JSON list on failure

In [3]:
products_and_category = utils.get_products_and_category()
products_and_category

{'Computers and Laptops': ['TechPro Ultrabook',
  'BlueWave Gaming Laptop',
  'PowerLite Convertible',
  'TechPro Desktop',
  'BlueWave Chromebook'],
 'Smartphones and Accessories': ['SmartX ProPhone',
  'MobiTech PowerCase',
  'SmartX MiniPhone',
  'MobiTech Wireless Charger',
  'SmartX EarBuds'],
 'Televisions and Home Theater Systems': ['CineView 4K TV',
  'SoundMax Home Theater',
  'CineView 8K TV',
  'SoundMax Soundbar',
  'CineView OLED TV'],
 'Gaming Consoles and Accessories': ['GameSphere X',
  'ProGamer Controller',
  'GameSphere Y',
  'ProGamer Racing Wheel',
  'GameSphere VR Headset'],
 'Audio Equipment': ['AudioPhonic Noise-Canceling Headphones',
  'WaveSound Bluetooth Speaker',
  'AudioPhonic True Wireless Earbuds',
  'WaveSound Soundbar',
  'AudioPhonic Turntable'],
 'Cameras and Camcorders': ['FotoSnap DSLR Camera',
  'ActionCam 4K',
  'FotoSnap Mirrorless Camera',
  'ZoomMaster Camcorder',
  'FotoSnap Instant Camera']}

### Find relevant product and category names (version 1)
This could be the version that is running in production.

In [4]:
def find_category_and_product_v1(user_input,products_and_category):

    delimiter = "####"
    system_message = f"""
    You will be provided with customer service queries. \
    The customer service query will be delimited with {delimiter} characters.
    Output only a valid JSON array, where each object has the following format:
        'category': <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems, \
    Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
    AND
        'products': <a list of products that must be found in the allowed products below>


    Where the categories and products must be found in the customer service query.
    If a product is mentioned, it must be associated with the correct category in the allowed products list below.
    If no products or categories are found, output an empty list.
    

    List out all products that are relevant to the customer service query based on how closely it relates
    to the product name and product category.
    Do not assume, from the name of the product, any features or attributes such as relative quality or price.

    The allowed products are provided in JSON format.
    The keys of each item represent the category.
    The values of each item is a list of products that are within that category.
    Allowed products: {products_and_category}
    

    """
    
    few_shot_user_1 = """I want the most expensive computer."""
    few_shot_assistant_1 = """ 
    [{'category': 'Computers and Laptops', \
'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
    """
    
    messages =  [  
    {'role':'system', 'content': system_message},    
    {'role':'user', 'content': f"{delimiter}{few_shot_user_1}{delimiter}"},  
    {'role':'assistant', 'content': few_shot_assistant_1 },
    {'role':'user', 'content': f"{delimiter}{user_input}{delimiter}"},  
    ] 
    return get_completion_from_messages(messages)


### Evaluate on some queries

In [5]:
customer_msg_0 = f"""Which TV can I buy if I'm on a budget?"""

products_by_category_0 = find_category_and_product_v1(customer_msg_0,
                                                      products_and_category)
print(products_by_category_0)

[
      {
        "category": "Televisions and Home Theater Systems",
        "products": ["SoundMax Soundbar"]
      }
    ]


In [6]:
customer_msg_1 = f"""I need a charger for my smartphone"""

products_by_category_1 = find_category_and_product_v1(customer_msg_1,
                                                      products_and_category)
print(products_by_category_1)

[
  {
    "category": "Smartphones and Accessories",
    "products": ["MobiTech PowerCase", "MobiTech Wireless Charger", "SmartX EarBuds"]
  }
]


In [7]:
customer_msg_2 = f"""
What computers do you have?"""

products_by_category_2 = find_category_and_product_v1(customer_msg_2,
                                                      products_and_category)
products_by_category_2

'[\n  {\n    "category": "Computers and Laptops",\n    "products": [\n      "TechPro Ultrabook",\n      "BlueWave Gaming Laptop",\n      "PowerLite Convertible",\n      "TechPro Desktop",\n      "BlueWave Chromebook"\n    ]\n  }\n]'

In [8]:
customer_msg_3 = f"""
tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs do you have?"""

products_by_category_3 = find_category_and_product_v1(customer_msg_3,
                                                      products_and_category)
print(products_by_category_3)

[
      {
        "category": "Smartphones and Accessories",
        "products": ["SmartX ProPhone", "MobiTech PowerCase", "SmartX MiniPhone", "MobiTech Wireless Charger", "SmartX EarBuds"]
      },
      {
        "category": "Cameras and Camcorders",
        "products": ["FotoSnap DSLR Camera", "ActionCam 4K", "FotoSnap Mirrorless Camera", "ZoomMaster Camcorder", "FotoSnap Instant Camera"]
      },
      {
        "category": "Televisions and Home Theater Systems",
        "products": ["CineView 4K TV", "SoundMax Home Theater", "CineView 8K TV", "SoundMax Soundbar", "CineView OLED TV"]
      }
    ]


### Harder test cases
Identify queries found in production, where the model is not working as expected.

In [9]:
customer_msg_4 = f"""
tell me about the CineView TV, the 8K one, Gamesphere console, the X one.
I'm on a budget, what computers do you have?"""

products_by_category_4 = find_category_and_product_v1(customer_msg_4,
                                                      products_and_category)
print(products_by_category_4)

[
  {
    "category": "Televisions and Home Theater Systems",
    "products": ["CineView 4K TV", "SoundMax Home Theater", "CineView 8K TV", "SoundMax Soundbar", "CineView OLED TV"]
  },
  {
    "category": "Gaming Consoles and Accessories",
    "products": ["GameSphere X", "ProGamer Controller", "GameSphere Y", "ProGamer Racing Wheel", "GameSphere VR Headset"]
  }
]
 
 
Let me know if you'd like more information about the computers, and I can give you a list of budget-friendly options.


### Modify the prompt to work on the hard test cases

In [10]:
def find_category_and_product_v2(user_input,products_and_category):
    """
    Added: Do not output any additional text that is not in JSON format.
    Added a second example (for few-shot prompting) where user asks for 
    the cheapest computer. In both few-shot examples, the shown response 
    is the full list of products in JSON only.
    """
    delimiter = "####"
    system_message = f"""
    You will be provided with customer service queries. \
    The customer service query will be delimited with {delimiter} characters.
    Output only a valid JSON array, where each object has the following format:
        'category': <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems, \
    Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
    AND
        'products': <a list of products that must be found in the allowed products below>
    Do not output any additional text that is not in JSON format.
    Do not write any explanatory text after outputting the requested JSON.


    Where the categories and products must be found in the customer service query.
    If a product is mentioned, it must be associated with the correct category in the allowed products list below.
    If no products or categories are found, output an empty list.
    

    List out all products that are relevant to the customer service query based on how closely it relates
    to the product name and product category.
    Do not assume, from the name of the product, any features or attributes such as relative quality or price.

    The allowed products are provided in JSON format.
    The keys of each item represent the category.
    The values of each item is a list of products that are within that category.
    Allowed products: {products_and_category}
    

    """
    
    few_shot_user_1 = """I want the most expensive computer. What do you recommend?"""
    few_shot_assistant_1 = """ 
    [{'category': 'Computers and Laptops', \
'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
    """
    
    few_shot_user_2 = """I want the most cheapest computer. What do you recommend?"""
    few_shot_assistant_2 = """ 
    [{'category': 'Computers and Laptops', \
'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
    """
    
    messages =  [  
    {'role':'system', 'content': system_message},    
    {'role':'user', 'content': f"{delimiter}{few_shot_user_1}{delimiter}"},  
    {'role':'assistant', 'content': few_shot_assistant_1 },
    {'role':'user', 'content': f"{delimiter}{few_shot_user_2}{delimiter}"},  
    {'role':'assistant', 'content': few_shot_assistant_2 },
    {'role':'user', 'content': f"{delimiter}{user_input}{delimiter}"},  
    ]
    
    response = get_completion_from_messages(messages)
    
    # Remove accidental extra closing bracket
    response = response.strip()
    
    if response.endswith("]\n    ]"):
        response = response[:-1].strip()
    
    return response


### Evaluate the modified prompt on the hard tests cases

In [11]:
customer_msg_3 = f"""
tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs do you have?"""

products_by_category_3 = find_category_and_product_v2(customer_msg_3,
                                                      products_and_category)
print(products_by_category_3)

[{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone', 'MobiTech PowerCase', 'SmartX MiniPhone', 'MobiTech Wireless Charger', 'SmartX EarBuds']},
     {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera', 'ActionCam 4K', 'FotoSnap Mirrorless Camera', 'ZoomMaster Camcorder', 'FotoSnap Instant Camera']},
     {'category': 'Televisions and Home Theater Systems', 'products': ['CineView 4K TV', 'SoundMax Home Theater', 'CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV']}]


### Regression testing: verify that the model still works on previous test cases
Check that modifying the model to fix the hard test cases does not negatively affect its performance on previous test cases.

In [12]:
customer_msg_0 = f"""Which TV can I buy if I'm on a budget?"""

products_by_category_0 = find_category_and_product_v2(customer_msg_0,
                                                      products_and_category)
print(products_by_category_0)

[{'category': 'Televisions and Home Theater Systems', 'products': ['CineView 4K TV', 'SoundMax Home Theater', 'CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV']}]


### Gather development set for automated testing

In [13]:
msg_ideal_pairs_set = [
    
    # eg 0
    {'customer_msg':"""Which TV can I buy if I'm on a budget?""",
     'ideal_answer':{
        'Televisions and Home Theater Systems':set(
            ['CineView 4K TV', 'SoundMax Home Theater', 'CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV']
        )}
    },

    # eg 1
    {'customer_msg':"""I need a charger for my smartphone""",
     'ideal_answer':{
        'Smartphones and Accessories':set(
            ['MobiTech PowerCase', 'MobiTech Wireless Charger', 'SmartX EarBuds']
        )}
    },
    # eg 2
    {'customer_msg':f"""What computers do you have?""",
     'ideal_answer':{
           'Computers and Laptops':set(
               ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook'
               ])
                }
    },

    # eg 3
    {'customer_msg':f"""tell me about the smartx pro phone and \
    the fotosnap camera, the dslr one.\
    Also, what TVs do you have?""",
     'ideal_answer':{
        'Smartphones and Accessories':set(
            ['SmartX ProPhone']),
        'Cameras and Camcorders':set(
            ['FotoSnap DSLR Camera']),
        'Televisions and Home Theater Systems':set(
            ['CineView 4K TV', 'SoundMax Home Theater','CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV'])
        }
    }, 
    
    # eg 4
    {'customer_msg':"""tell me about the CineView TV, the 8K one, Gamesphere console, the X one.
I'm on a budget, what computers do you have?""",
     'ideal_answer':{
        'Televisions and Home Theater Systems':set(
            ['CineView 8K TV']),
        'Gaming Consoles and Accessories':set(
            ['GameSphere X']),
        'Computers and Laptops':set(
            ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook'])
        }
    },
    
    # eg 5
    {'customer_msg':f"""What smartphones do you have?""",
     'ideal_answer':{
           'Smartphones and Accessories':set(
               ['SmartX ProPhone', 'MobiTech PowerCase', 'SmartX MiniPhone', 'MobiTech Wireless Charger', 'SmartX EarBuds'
               ])
                    }
    },
    # eg 6
    {'customer_msg':f"""I'm on a budget.  Can you recommend some smartphones to me?""",
     'ideal_answer':{
        'Smartphones and Accessories':set(
            ['SmartX EarBuds', 'SmartX MiniPhone', 'MobiTech PowerCase', 'SmartX ProPhone', 'MobiTech Wireless Charger']
        )}
    },

    # eg 7 # this will output a subset of the ideal answer
    {'customer_msg':f"""What Gaming consoles would be good for my friend who is into racing games?""",
     'ideal_answer':{
        'Gaming Consoles and Accessories':set([
            'GameSphere X',
            'ProGamer Controller',
            'GameSphere Y',
            'ProGamer Racing Wheel',
            'GameSphere VR Headset'
     ])}
    },
    # eg 8
    {'customer_msg':f"""What could be a good present for my videographer friend?""",
     'ideal_answer': {
        'Cameras and Camcorders':set([
        'FotoSnap DSLR Camera', 'ActionCam 4K', 'FotoSnap Mirrorless Camera', 'ZoomMaster Camcorder', 'FotoSnap Instant Camera'
        ])}
    },
    
    # eg 9
    {'customer_msg':f"""I would like a hot tub time machine.""",
     'ideal_answer': []
    }
    
]


### Evaluate test cases by comparing to the ideal answers

In [14]:
import json
# This function compares the model's response
# with the ideal/expected answer and returns a score
def eval_response_with_ideal(response,
                              ideal,
                              debug=False):

    # If debug is True, print the model's response
    if debug:
        print("response")
        print(response)
    
    # json.loads() expects double quotes instead of single quotes
    # So, replace single quotes with double quotes
    json_like_str = response.replace("'",'"')
    
    # Convert the JSON-like string into a Python list of dictionaries
    l_of_d = json.loads(json_like_str)
    
    # Special case:
    # If both the response and ideal answer are empty,
    # the response is considered completely correct
    if l_of_d == [] and ideal == []:
        return 1
    
    # If only one of them is empty,
    # then there is a mismatch, so return 0
    elif l_of_d == [] or ideal == []:
        return 0
    # Keep track of how many answers are correct    
    correct = 0    
    # If debug is True, print the parsed response    
    if debug:
        print("l_of_d is")
        print(l_of_d)
    # Go through each dictionary in the response        
    for d in l_of_d:
        # Get the category from the response
        cat = d.get('category')
        # Get the list of products from the response
        prod_l = d.get('products')
        # Continue only if both category and products exist
        if cat and prod_l:
            # convert list to set for comparison
            prod_set = set(prod_l)
            # Get the ideal/expected products for this category
            ideal_cat = ideal.get(cat)
            # If the category exists in the ideal answer
            if ideal_cat:
                # Convert the ideal product list into a set
                prod_set_ideal = set(ideal.get(cat))
            # If the category does not exist in the ideal answer    
            else:
                if debug:
                    print(f"did not find category {cat} in ideal")
                    print(f"ideal: {ideal}")
                # Skip this response and move to the next one
                continue
            # Print the sets when debugging is enabled    
            if debug:
                print("prod_set\n",prod_set)
                print()
                print("prod_set_ideal\n",prod_set_ideal)
            # Compare the model's products with the ideal products
            if prod_set == prod_set_ideal:
                # The response is correct
                if debug:
                    print("correct")
                # Increase the correct count
                correct +=1
            else:
                # The response is not exactly correct
                print("incorrect")
                print(f"prod_set: {prod_set}")
                print(f"prod_set_ideal: {prod_set_ideal}")
                # Check if the response contains fewer products
                # than the ideal answer
                if prod_set <= prod_set_ideal:
                    print("response is a subset of the ideal answer")
                # Check if the response contains more products
                # than the ideal answer
                elif prod_set >= prod_set_ideal:
                    print("response is a superset of the ideal answer")

    # Calculate the fraction of correct responses
    # by dividing correct answers by total responses
    pc_correct = correct / len(l_of_d)
    # Return the final score        
    return pc_correct

In [15]:
print(f'Customer message: {msg_ideal_pairs_set[7]["customer_msg"]}')
print(f'Ideal answer: {msg_ideal_pairs_set[7]["ideal_answer"]}')


Customer message: What Gaming consoles would be good for my friend who is into racing games?
Ideal answer: {'Gaming Consoles and Accessories': {'GameSphere X', 'ProGamer Controller', 'GameSphere VR Headset', 'ProGamer Racing Wheel', 'GameSphere Y'}}


In [16]:
response = find_category_and_product_v2(msg_ideal_pairs_set[7]["customer_msg"],
                                         products_and_category)
print(f'Resonse: {response}')

eval_response_with_ideal(response,
                              msg_ideal_pairs_set[7]["ideal_answer"])

Resonse: [{'category': 'Gaming Consoles and Accessories', 'products': ['GameSphere X', 'ProGamer Controller', 'GameSphere Y', 'ProGamer Racing Wheel', 'GameSphere VR Headset']}]


1.0

### Run evaluation on all test cases and calculate the fraction of cases that are correct

In [17]:
# Note, this will not work if any of the api calls time out
score_accum = 0
# Go through each customer message and its ideal answer
for i, pair in enumerate(msg_ideal_pairs_set):
    print(f"example {i}")
    # Get the customer's message
    customer_msg = pair['customer_msg']
    # Get the expected/ideal answer
    ideal = pair['ideal_answer']
    
    # Send the customer message to the function
    # to get the model's response
    response = find_category_and_product_v2(customer_msg,
                                                      products_and_category)
    
    
    # Compare the model's response with the ideal answer
    # and calculate a score
    score = eval_response_with_ideal(response,ideal,debug=False)
    print(f"{i}: {score}")
    # Add the score to the total accumulated score
    score_accum += score
    
# Count the total number of examples
n_examples = len(msg_ideal_pairs_set)
# Calculate the average score / fraction correct
fraction_correct = score_accum / n_examples
# Print the final result
print(f"Fraction correct out of {n_examples}: {fraction_correct}")

example 0
incorrect
prod_set: {'SoundMax Soundbar', 'CineView 4K TV'}
prod_set_ideal: {'CineView OLED TV', 'SoundMax Soundbar', 'CineView 8K TV', 'SoundMax Home Theater', 'CineView 4K TV'}
response is a subset of the ideal answer
0: 0.0
example 1
1: 1.0
example 2
2: 1.0
example 3
incorrect
prod_set: {'SmartX EarBuds', 'SmartX MiniPhone', 'MobiTech Wireless Charger', 'MobiTech PowerCase', 'SmartX ProPhone'}
prod_set_ideal: {'SmartX ProPhone'}
response is a superset of the ideal answer
incorrect
prod_set: {'ActionCam 4K', 'FotoSnap Mirrorless Camera', 'ZoomMaster Camcorder', 'FotoSnap Instant Camera', 'FotoSnap DSLR Camera'}
prod_set_ideal: {'FotoSnap DSLR Camera'}
response is a superset of the ideal answer
3: 0.3333333333333333
example 4
incorrect
prod_set: {'SoundMax Soundbar', 'CineView OLED TV', 'CineView 8K TV', 'SoundMax Home Theater', 'CineView 4K TV'}
prod_set_ideal: {'CineView 8K TV'}
response is a superset of the ideal answer
incorrect
prod_set: {'GameSphere X', 'ProGamer Contr